In [5]:
"""
Lightweight Swin-Style Windowed Transformer Neck for ADAS Object Detection
Member 2 Contribution: Fuses CNN multi-scale features with global context.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / (window_size * window_size)))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class WindowAttention(nn.Module):
    """Window-based Multi-Head Self-Attention (W-MSA) with relative position bias."""
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads)
        )

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1
        ).permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class SwinTransformerBlock(nn.Module):
    """Swin Transformer Block with Shifted Window Self-Attention."""
    def __init__(self, dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4.0, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim, window_size=(self.window_size, self.window_size), num_heads=num_heads,
            attn_drop=attn_drop, proj_drop=drop
        )

        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden_dim, dim),
            nn.Dropout(drop)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).contiguous()

        pad_h = (self.window_size - H % self.window_size) % self.window_size
        pad_w = (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        
        Hp, Wp = H + pad_h, W + pad_w

        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)

        attn_windows = self.attn(self.norm1(x_windows))

        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, Hp, Wp)

        if self.shift_size > 0:
            x_out = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x_out = shifted_x

        if pad_h > 0 or pad_w > 0:
            x_out = x_out[:, :H, :W, :].contiguous()

        x = x + x_out
        x = x + self.mlp(self.norm2(x))
        return x.permute(0, 3, 1, 2).contiguous()


class LightweightTransformerNeck(nn.Module):
    """Fuses multi-scale CNN backbone feature maps P3, P4, P5 with Swin attention."""
    def __init__(self, in_channels=[256, 512, 1024], out_channels=256, num_blocks=2, window_size=7):
        super().__init__()
        self.p3_proj = nn.Conv2d(in_channels[0], out_channels, kernel_size=1)
        self.p4_proj = nn.Conv2d(in_channels[1], out_channels, kernel_size=1)
        self.p5_proj = nn.Conv2d(in_channels[2], out_channels, kernel_size=1)

        self.transformer_p3 = nn.ModuleList([
            SwinTransformerBlock(out_channels, num_heads=4, window_size=window_size, shift_size=0 if i % 2 == 0 else window_size // 2)
            for i in range(num_blocks)
        ])
        self.transformer_p4 = nn.ModuleList([
            SwinTransformerBlock(out_channels, num_heads=4, window_size=window_size, shift_size=0 if i % 2 == 0 else window_size // 2)
            for i in range(num_blocks)
        ])
        self.transformer_p5 = nn.ModuleList([
            SwinTransformerBlock(out_channels, num_heads=4, window_size=window_size, shift_size=0 if i % 2 == 0 else window_size // 2)
            for i in range(num_blocks)
        ])

        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.smooth_p4 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth_p3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, features):
        P3, P4, P5 = features
        P3_p, P4_p, P5_p = self.p3_proj(P3), self.p4_proj(P4), self.p5_proj(P5)

        for blk in self.transformer_p5:
            P5_p = blk(P5_p)
        
        P4_fused = P4_p + self.upsample(P5_p)
        for blk in self.transformer_p4:
            P4_fused = blk(P4_fused)
        P4_out = self.smooth_p4(P4_fused)

        P3_fused = P3_p + self.upsample(P4_out)
        for blk in self.transformer_p3:
            P3_fused = blk(P3_fused)
        P3_out = self.smooth_p3(P3_fused)

        return [P3_out, P4_out, P5_p]

In [6]:
"""
Complete Hybrid CNN–Vision Transformer Architecture for ADAS Detection
"""

import torch
import torch.nn as nn
from models.transformer_neck import LightweightTransformerNeck


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class CSPBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_bottlenecks=1):
        super().__init__()
        hidden = out_channels // 2
        self.cv1 = ConvBlock(in_channels, hidden, 1, 1, 0)
        self.cv2 = ConvBlock(in_channels, hidden, 1, 1, 0)
        self.bottlenecks = nn.Sequential(*[
            nn.Sequential(
                ConvBlock(hidden, hidden, 3, 1, 1),
                ConvBlock(hidden, hidden, 3, 1, 1)
            ) for _ in range(num_bottlenecks)
        ])
        self.cv3 = ConvBlock(hidden * 2, out_channels, 1, 1, 0)

    def forward(self, x):
        return self.cv3(torch.cat((self.bottlenecks(self.cv1(x)), self.cv2(x)), dim=1))


class CSPDarknetBackbone(nn.Module):
    def __init__(self, in_channels=3, channel_list=[64, 128, 256, 512]):
        super().__init__()
        self.stem = ConvBlock(in_channels, channel_list[0], 3, 2, 1)
        self.stage1 = nn.Sequential(
            ConvBlock(channel_list[0], channel_list[1], 3, 2, 1),
            CSPBlock(channel_list[1], channel_list[1], num_bottlenecks=1)
        )
        self.stage_p3 = nn.Sequential(
            ConvBlock(channel_list[1], channel_list[2], 3, 2, 1),  # Stride 8
            CSPBlock(channel_list[2], channel_list[2], num_bottlenecks=2)
        )
        self.stage_p4 = nn.Sequential(
            ConvBlock(channel_list[2], channel_list[3], 3, 2, 1),  # Stride 16
            CSPBlock(channel_list[3], channel_list[3], num_bottlenecks=2)
        )
        self.stage_p5 = nn.Sequential(
            ConvBlock(channel_list[3], channel_list[3] * 2, 3, 2, 1),  # Stride 32
            CSPBlock(channel_list[3] * 2, channel_list[3] * 2, num_bottlenecks=1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        p3 = self.stage_p3(x)   # Channels: 256
        p4 = self.stage_p4(p3)  # Channels: 512
        p5 = self.stage_p5(p4)  # Channels: 1024
        return [p3, p4, p5]


class DecoupledHead(nn.Module):
    def __init__(self, in_channels=256, num_classes=10):
        super().__init__()
        self.cls_conv = nn.Sequential(
            ConvBlock(in_channels, in_channels, 3, 1, 1),
            ConvBlock(in_channels, in_channels, 3, 1, 1),
            nn.Conv2d(in_channels, num_classes, kernel_size=1)
        )
        self.reg_conv = nn.Sequential(
            ConvBlock(in_channels, in_channels, 3, 1, 1),
            ConvBlock(in_channels, in_channels, 3, 1, 1),
            nn.Conv2d(in_channels, 4, kernel_size=1)
        )

    def forward(self, x):
        return self.cls_conv(x), self.reg_conv(x)


class HybridDetector(nn.Module):
    def __init__(self, num_classes=10, num_transformer_blocks=2, neck_channels=256):
        super().__init__()
        self.backbone = CSPDarknetBackbone(in_channels=3, channel_list=[64, 128, 256, 512])
        self.neck = LightweightTransformerNeck(
            in_channels=[256, 512, 1024],
            out_channels=neck_channels,
            num_blocks=num_transformer_blocks,
            window_size=7
        )
        self.head_p3 = DecoupledHead(in_channels=neck_channels, num_classes=num_classes)
        self.head_p4 = DecoupledHead(in_channels=neck_channels, num_classes=num_classes)
        self.head_p5 = DecoupledHead(in_channels=neck_channels, num_classes=num_classes)

    def forward(self, x):
        p3, p4, p5 = self.backbone(x)
        n3, n4, n5 = self.neck([p3, p4, p5])
        cls_p3, reg_p3 = self.head_p3(n3)
        cls_p4, reg_p4 = self.head_p4(n4)
        cls_p5, reg_p5 = self.head_p5(n5)

        return {
            "p3": {"cls": cls_p3, "reg": reg_p3},
            "p4": {"cls": cls_p4, "reg": reg_p4},
            "p5": {"cls": cls_p5, "reg": reg_p5}
        }

ModuleNotFoundError: No module named 'models'

In [ ]:
"""
Training Engine for Hybrid CNN–Vision Transformer ADAS Detector
Member 1 & 2 Contribution: Optimized for Kaggle / Google Colab T4 GPUs
"""

import os
import time
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from models.hybrid_detector import HybridDetector


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--num-blocks", type=int, default=2)
    parser.add_argument("--save-dir", type=str, default="./checkpoints")
    args = parser.parse_args()

    os.makedirs(args.save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[+] Active Device: {device}")

    model = HybridDetector(num_classes=10, num_transformer_blocks=args.num_blocks).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    print(f"[+] Loaded HybridDetector ({args.num_blocks} Transformer Blocks)")
    # Execution loop runs with automatic mixed precision on GPU environments

In [ ]:
"""
Ablation Study Script: Accuracy vs. Speed Trade-Off Benchmark
Member 1 & 2 Contribution: Evaluates 1, 2, and 4 Transformer Blocks
"""

import time
import torch
from models.hybrid_detector import HybridDetector


def measure_inference_speed(model, input_tensor, num_runs=50):
    model.eval()
    with torch.no_grad():
        for _ in range(10):  # Warmup
            _ = model(input_tensor)

    start = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(input_tensor)
            if input_tensor.is_cuda:
                torch.cuda.synchronize()

    latency_ms = ((time.time() - start) / num_runs) * 1000.0
    return latency_ms, 1000.0 / latency_ms


def run_ablation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dummy = torch.randn(1, 3, 640, 640).to(device)

    print(f"\n{'Block Config':<20} | {'Params (M)':<12} | {'Latency (ms)':<15} | {'FPS':<10}")
    print("-" * 65)

    for num_blocks in [1, 2, 4]:
        model = HybridDetector(num_classes=10, num_transformer_blocks=num_blocks).to(device)
        params = sum(p.numel() for p in model.parameters()) / 1e6
        lat, fps = measure_inference_speed(model, dummy)
        print(f"Hybrid ({num_blocks} Block{'s' if num_blocks > 1 else ''})   | {params:<12.2f} | {lat:<15.2f} | {fps:<10.1f}")


if __name__ == "__main__":
    run_ablation()